# 02 - Modelado Predictivo y Evaluación de Clasificación

**Asignatura:** Programación Avanzada en Ciencia de Datos  
**Problema:** Tipificación y Segmentación de Propiedades en CABA (Gama Alta vs. Estándar)  

Este notebook documenta el proceso experimental conforme a la consigna oficial:
1. **Preprocesamiento:** Pipeline con `ColumnTransformer` (imputación, escalado estándar y One-Hot Encoding).
2. **Modelos:** Comparación de al menos dos algoritmos de clasificación (`LogisticRegression` y `RandomForestClassifier`).
3. **Optimización:** Búsqueda de hiperparámetros con `GridSearchCV` y validación cruzada estratificada.
4. **Evaluación:** Accuracy, Precision, Recall, F1-Score y Matriz de Confusión.
5. **Persistencia:** Almacenamiento del dataset limpio en SQLite y de resultados/parámetros en MongoDB.

In [ ]:
import sys
from pathlib import Path

# Añadir directorio raíz al path para importar módulos de src
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from config import YEARS, BASE_DIR, SQLITE_DB_PATH, SQLITE_TABLE_CLEAN, MONGO_URI, MONGO_DB_NAME
from src.data_loader import build_unified_dataset
from src.cleaning import clean_and_prepare_dataset
from src.features import build_preprocessor, split_features_and_target
from src.models import (
    create_dummy_classifier_pipeline,
    create_logistic_regression_pipeline,
    create_random_forest_pipeline,
    optimize_with_gridsearch,
    compare_models,
    save_model,
)
from src.evaluation import evaluate_classification, plot_and_save_confusion_matrix
from src.db_sql import save_clean_dataset
from src.db_mongo import save_model_results_mongo, save_model_parameters_mongo
from src.logger_setup import get_logger

logger = get_logger("02_modelado_notebook")
print("Entorno inicializado correctamente.")

## 1. Carga, Limpieza y Binarización del Target

In [ ]:
# Cargar datos crudos unificados de los 16 años
df_raw = build_unified_dataset(YEARS)

# Ejecutar limpieza, filtrado de outliers y target ES_GAMA_ALTA
df_clean = clean_and_prepare_dataset(df_raw, quantile_threshold=0.75)

# Persistir exclusivamente el dataset limpio en SQLite (inmuebles.db)
save_clean_dataset(df_clean, SQLITE_DB_PATH, SQLITE_TABLE_CLEAN)
df_clean.head()

## 2. Partición de Datos (Train / Test Split)

In [ ]:
X, y = split_features_and_target(df_clean, target_column="ES_GAMA_ALTA")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Dimensiones entrenamiento: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Dimensiones prueba: X_test={X_test.shape}, y_test={y_test.shape}")

## 3. Modelo 1: Regresión Logística (Baseline Lineal)

In [ ]:
preprocessor = build_preprocessor()

# Modelo 0: Baseline Ingenuo (DummyClassifier)
pipe_dummy = create_dummy_classifier_pipeline(preprocessor, strategy='most_frequent')
pipe_dummy.fit(X_train, y_train)

# Modelo 1: Regresión Logística (Baseline Lineal)
pipe_lr = create_logistic_regression_pipeline(preprocessor)
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
metrics_lr = evaluate_classification(y_test, y_pred_lr)
print("Métricas Regresión Logística:", {k: v for k, v in metrics_lr.items() if k not in ['confusion_matrix', 'classification_report']})

## 4. Modelo 2: Random Forest con GridSearchCV

In [ ]:
pipe_rf = create_random_forest_pipeline(preprocessor)

param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [10, 20],
}

# Subconjunto estratificado para optimización rápida
sample_idx = X_train.sample(n=min(30000, len(X_train)), random_state=42).index
grid_search = optimize_with_gridsearch(
    pipe_rf, param_grid, X_train.loc[sample_idx], y_train.loc[sample_idx], cv_splits=3
)

# Reentrenar el mejor pipeline con todo el conjunto de entrenamiento
best_rf = grid_search.best_estimator_
best_rf.fit(X_train, y_train)

## 5. Comparación de Modelos y Selección del Ganador

In [ ]:
candidatos = {
    "DummyClassifier": pipe_dummy,
    "LogisticRegression": pipe_lr,
    "RandomForest_Tuned": best_rf,
}

winner_name, winner_pipeline, all_metrics = compare_models(candidatos, X_test, y_test)

# Tabla comparativa de métricas
df_comparativo = pd.DataFrame([
    {"Modelo": name, "Accuracy": m["accuracy"], "Precision": m["precision"], "Recall": m["recall"], "F1-Score": m["f1_score"]}
    for name, m in all_metrics.items()
])
display(df_comparativo)

## 6. Matriz de Confusión del Modelo Ganador

In [ ]:
y_pred_winner = winner_pipeline.predict(X_test)
plot_and_save_confusion_matrix(
    y_test,
    y_pred_winner,
    output_path=BASE_DIR / "logs" / "matriz_confusion_ganador.png",
    title=f"Matriz de Confusión - {winner_name}"
)
plt.show()

## 7. Persistencia de Resultados y Parámetros en MongoDB

In [ ]:
y_prob_winner = winner_pipeline.predict_proba(X_test) if hasattr(winner_pipeline, "predict_proba") else None

# Guardar resultados y métricas en MongoDB (colección 'resultados')
save_model_results_mongo(
    y_true=y_test,
    y_pred=y_pred_winner,
    y_prob=y_prob_winner,
    metrics=all_metrics[winner_name],
    model_name=winner_name,
    uri=MONGO_URI,
    db_name=MONGO_DB_NAME,
)

# Guardar hiperparámetros en MongoDB (colección 'parametros')
save_model_parameters_mongo(
    model_name=winner_name,
    best_params=grid_search.best_params_,
    uri=MONGO_URI,
    db_name=MONGO_DB_NAME,
)

# Serializar modelo en disco para la app Streamlit
save_model(winner_pipeline, BASE_DIR / "app" / "modelo_ganador.joblib")
print("Proceso de modelado y persistencia en MongoDB finalizado con éxito.")